# 11B · The Honest Backtest — What Survives Contact With Reality
### Financial Analytics — Module 11 · Lab 3

The moment 9B made you wait for. The 50/200 MA crossover — "hold when the fast average is above the slow, step aside otherwise" — finally gets **traded**, under this course's full discipline:

1. Signals **lagged one day** (the look-ahead vaccine, in code)
2. **Costs charged** on every switch (brokerage + taxes + 11A's impact)
3. Judged against the honest floor: **buy-and-hold**
4. Tested across the **whole universe**, not one cherry-picked chart
5. And the registry's **planted landmine** — find it before it finds you

> 🛡️ **Bias check (LAW):**
> **Look-ahead** — a crossover computed from today's close cannot be traded AT today's close; we trade at the NEXT day's price. Enforced with `.shift(1)`. ☐
> **Survivorship** — our 20-stock universe is today's large caps: a survivor set. Results are OPTIMISTIC by construction; stated. ☐
> **Point-in-time** — prices only, no restated fundamentals. ☐
> **Regime** — sample spans calm and choppy regimes (good); one bull-heavy period (stated). ☐

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

import os
BASE = "data/" if os.path.exists("data") else "https://raw.githubusercontent.com/vivekhashtag/financial-analytics-course/main/data/"
uni = pd.read_csv(BASE + "nse_stock_universe.csv", parse_dates=["date"])
px = uni.pivot(index="date", columns="ticker", values="close").sort_index()
print(px.shape, "| tickers:", len(px.columns))

---
## 1. The engine — 20 readable lines

Study every line. The `.shift(1)` is the one that separates an honest backtest from a fantasy: the position we HOLD today was DECIDED yesterday.

In [ ]:
def backtest_crossover(close, fast=50, slow=200, cost_bps=15):
    """Long when fast MA > slow MA (decided on yesterday's data), else in cash.
    cost_bps charged on every position CHANGE (entry or exit).
    Returns a dict of results. cost_bps=15 ~ brokerage+STT+slippage for a liquid large-cap."""
    rets = close.pct_change()

    signal = (close.rolling(fast).mean() > close.rolling(slow).mean()).astype(int)
    position = signal.shift(1)                      # <- THE LINE. Yesterday's signal, today's position.

    trades = position.diff().abs().fillna(0)        # 1 on every switch
    strat_rets = position*rets - trades*cost_bps/10_000

    equity_strat = (1 + strat_rets.fillna(0)).cumprod()
    equity_hold  = (1 + rets.fillna(0)).cumprod()

    def stats(eq, r):
        yrs = len(r)/252
        cagr = eq.iloc[-1]**(1/yrs) - 1
        sharpe = r.mean()/r.std()*np.sqrt(252) if r.std() > 0 else np.nan
        maxdd = (eq/eq.cummax() - 1).min()
        return cagr, sharpe, maxdd

    c_s, s_s, d_s = stats(equity_strat, strat_rets.dropna())
    c_h, s_h, d_h = stats(equity_hold, rets.dropna())
    return {"cagr_strat": c_s, "sharpe_strat": s_s, "maxdd_strat": d_s,
            "cagr_hold": c_h, "sharpe_hold": s_h, "maxdd_hold": d_h,
            "n_trades": int(trades.sum()),
            "equity_strat": equity_strat, "equity_hold": equity_hold}

In [ ]:
# First flight: one liquid large-cap, gross vs net of costs
r_gross = backtest_crossover(px["RELIANCE.NS"], cost_bps=0)
r_net   = backtest_crossover(px["RELIANCE.NS"], cost_bps=15)

print(f"{'RELIANCE.NS':<14} {'CAGR':>8} {'Sharpe':>8} {'MaxDD':>8} {'trades':>7}")
print(f"{'buy & hold':<14} {r_net['cagr_hold']:>8.1%} {r_net['sharpe_hold']:>8.2f} {r_net['maxdd_hold']:>8.1%} {'-':>7}")
print(f"{'crossover g':<14} {r_gross['cagr_strat']:>8.1%} {r_gross['sharpe_strat']:>8.2f} {r_gross['maxdd_strat']:>8.1%} {r_gross['n_trades']:>7}")
print(f"{'crossover net':<14} {r_net['cagr_strat']:>8.1%} {r_net['sharpe_strat']:>8.2f} {r_net['maxdd_strat']:>8.1%} {r_net['n_trades']:>7}")
print(f"\nCost drag: {(r_gross['cagr_strat']-r_net['cagr_strat'])*100:.2f} pp of CAGR - every switch pays the 11A toll.")

---
## 2. The universe test — and the landmine

One chart proves nothing (you can always find a stock where any rule worked). Run the identical rule across all 20:

In [ ]:
rows = []
for t in px.columns:
    r = backtest_crossover(px[t].dropna())
    rows.append({"ticker": t, "strat_cagr": r["cagr_strat"], "hold_cagr": r["cagr_hold"],
                 "edge": r["cagr_strat"]-r["cagr_hold"], "trades": r["n_trades"],
                 "strat_dd": r["maxdd_strat"], "hold_dd": r["maxdd_hold"]})
board = pd.DataFrame(rows).set_index("ticker").sort_values("edge")
print(board.round(3).to_string())
print("\n>>> STOP. Look at the extreme rows before reading another word. Something is WRONG with one of them. <<<")

**Found it?** TATAMOTORS shows a buy-and-hold CAGR so catastrophically negative it would mean the company nearly ceased to exist — which it did not. Pull the thread:

In [ ]:
tm = px["TATAMOTORS.NS"].dropna()
worst_day = tm.pct_change().idxmin()
print(f"TATAMOTORS worst 'return': {tm.pct_change().min():.1%} on {worst_day.date()}")
print(tm.loc["2024-08-28":"2024-09-05"].round(1).to_string())
print("\nAn overnight -80% with no news? That is the registry's UNADJUSTED 1:5 STOCK SPLIT (2024-09-02):")
print("one share worth ~Rs 406 became five shares worth ~Rs 81 each. The shareholder lost NOTHING;")
print("the naive return calculation invented a crash. Module 1's adjusted-vs-unadjusted lesson, detonating")
print("inside a backtest exactly as promised. EVERY row of our leaderboard that touched this stock is garbage.")

In [ ]:
# The fix: back-adjust pre-split prices by the split ratio, then re-run
tm_adj = tm.copy()
tm_adj.loc[:"2024-09-01"] = tm_adj.loc[:"2024-09-01"] / 5

r_bad  = backtest_crossover(tm)
r_good = backtest_crossover(tm_adj)
print(f"{'TATAMOTORS':<12} {'hold CAGR':>10} {'strat CAGR':>11}")
print(f"{'unadjusted':<12} {r_bad['cagr_hold']:>10.1%} {r_bad['cagr_strat']:>11.1%}   <- fiction")
print(f"{'adjusted':<12} {r_good['cagr_hold']:>10.1%} {r_good['cagr_strat']:>11.1%}   <- reality")
print("\nOne corporate action, unhandled, and a backtest flips from disaster to normal.")
print("Professional shops run entire 'corporate actions' teams for exactly this reason.")

---
## 3. The honest verdict

In [ ]:
# Rebuild the leaderboard with TATAMOTORS fixed, then read it like an adult
px_fixed = px.copy()
px_fixed.loc[:"2024-09-01", "TATAMOTORS.NS"] = px_fixed.loc[:"2024-09-01", "TATAMOTORS.NS"] / 5

rows = []
for t in px_fixed.columns:
    r = backtest_crossover(px_fixed[t].dropna())
    rows.append({"ticker": t.replace(".NS",""), "edge_cagr": r["cagr_strat"]-r["cagr_hold"],
                 "dd_saved": r["maxdd_hold"]-r["maxdd_strat"], "trades": r["n_trades"]})
board = pd.DataFrame(rows).set_index("ticker")

fig, ax = plt.subplots(figsize=(10, 4))
colors = ["#16A34A" if v > 0 else "#DC2626" for v in board.sort_values("edge_cagr")["edge_cagr"]]
ax.barh(board.sort_values("edge_cagr").index, board.sort_values("edge_cagr")["edge_cagr"]*100, color=colors)
ax.axvline(0, color="black", lw=1)
ax.set_title(f"Crossover minus buy-and-hold, net of costs: beats hold on {(board.edge_cagr>0).mean():.0%} of stocks",
             loc="left", fontweight="bold")
ax.set_xlabel("CAGR edge (pp)"); plt.tight_layout(); plt.show()

print(f"Median CAGR edge : {board.edge_cagr.median()*100:+.2f} pp   (return: roughly a coin toss, minus costs)")
print(f"Median DD saved  : {board.dd_saved.median()*100:+.1f} pp   (risk: the strategy sat out the worst falls)")
print(f"Median trades    : {board.trades.median():.0f} over 4 years")

**Read the verdict the way Module 6 trained you.** On *returns*, the crossover is roughly a coin toss across the universe — and costs tilt the coin against you. Exactly what "prices eat their own forecasts" predicted. But the drawdown column tells a second story: the rule reliably **sat out the deepest falls** (it's mechanically out after long declines). So the honest conclusion is neither "it works" nor "it's garbage" — it's:

> *The crossover is not a return machine; it is a crude risk-reduction rule that pays for its insurance with whipsaw costs and missed rebounds. Whether that trade-off suits you is a risk-preference question, not a forecasting one.*

That nuanced sentence — not a P&L screenshot — is what an honest backtest produces.

### ✏️ Exercises
1. **The parameter graveyard:** grid-search fast∈{20,50}, slow∈{100,200} across the universe. Report the best combo's median edge — then re-read 6B Exercise 1 and write down why that number is now contaminated, and what protocol would clean it.
2. **Costs, the dial of death:** re-run the universe at 5, 15, 40 bps. At what cost level does the median edge die entirely? What does that say about who CAN run such strategies (hint: 11A's market maker pays ~0)?
3. **The regime cut:** split each stock's backtest at 2024-01-01. Does the crossover's edge live in one regime? Connect your answer to the Bias Check's fourth line.

---
## Lab 3 complete

You can read an order book, price your own market impact, explain the maker's trade and the speed race, and — the rarer skill — run a backtest that would survive an auditor: lagged signals, real costs, a universe not a cherry-pick, a landmine caught, and a verdict with nuance instead of a victory lap. **Badge: Honest Trader ⚖️**

*AI disclosure: ______*

In [ ]:
# workspace
